#### Now we'll split your 4000 training images into two groups:

1. Train set (80%, ~3200 images) — the model learns from this
2. Validation set (20%, ~800 images) — you hide the true labels from the model and check how well it guesses, to see if it's actually learning something real or just memorizing

Then, since this is multi-label (an X-ray can have 0, 1, or several conditions at once), the simplest and most standard classical-ML approach is: train 5 separate binary classifiers, one per condition — each just answers "does this image show Atelectasis: yes/no", independently for each of the 5 conditions.

In [1]:
%pip install xgboost


Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

In [3]:
# Load and merge features with labels

features = pd.read_csv('Outputs/train_features.csv')
labels = pd.read_csv('train_labels.csv')

df = features.merge(labels, left_on='filename', right_on='Image Index')

print("Merged shape:", df.shape)  # sanity check — should match features.shape[0]

conditions = ['Atelectasis', 'Effusion', 'Infiltration', 'Nodule', 'Pneumothorax']
feature_cols = [c for c in features.columns if c != 'filename']

# 3. Drop NaNs — must happen BEFORE X and y are created
df = df.dropna()
print("NaNs remaining:", df.isna().sum().sum())  # should print 0

X = df[feature_cols]
y = df[conditions]


Merged shape: (4484, 28)
NaNs remaining: 0


In [4]:
# Train/val split (80/20)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# confirm 21 columns, blob names present
print(X_train.shape, sorted(X_train.columns.tolist()))


(3548, 21) ['glcm_contrast', 'glcm_energy', 'glcm_homogeneity', 'hog_mean', 'hog_std', 'lower_kurtosis', 'lower_mean', 'lower_skew', 'lower_std', 'max_blob_size', 'mean_blob_size', 'middle_kurtosis', 'middle_mean', 'middle_skew', 'middle_std', 'num_blobs', 'symmetry_score', 'upper_kurtosis', 'upper_mean', 'upper_skew', 'upper_std']


####    Initially, there was an error with the code.

####    We get an error saying "Input X is Nan", i.e somewhere in the feature table, at least one cell is empty (NaN) — scikit-learn models can't handle missing values, so it refuses to train.

####    Then we found out that there are only 49/4484 rows that contain Na values.

####    As its a small number, the best thing to do is to drop the 49 NaN rows and document it as a limitaion.

In [5]:
#Displaying no. of rows with NaN values
print("Total NaNs per column:")
print(df.isna().sum()[df.isna().sum() > 0])
print("\nRows with any NaN:", df.isna().any(axis=1).sum(), "out of", len(df))

#First 5 rows with NaN values
df[df.isna().any(axis=1)].head()

#Drop the NaN rows.
before = len(df)
df = df.dropna()
after = len(df)
print(f"Dropped {before - after} rows with missing features")
print("\nRows with any NaN:", df.isna().any(axis=1).sum(), "out of", len(df))

#Nan values in X and Y. (Initially, X had NaN values, but now we have fixed the issue)
print("NaNs in X:", X.isna().sum().sum())
print("NaNs in y:", y.isna().sum().sum())

Total NaNs per column:
Series([], dtype: int64)

Rows with any NaN: 0 out of 4435
Dropped 0 rows with missing features

Rows with any NaN: 0 out of 4435
NaNs in X: 0
NaNs in y: 0


####    Another issue we faced was with the LogisticRegression convergence warning. It's not fatal, but it means LogisticRegression didn't fully finish optimizing — so its scores (which won for Nodule) might be understated rather than a fair fight against RandomForest/XGBoost

In [6]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_val_scaled = pd.DataFrame(scaler.transform(X_val), columns=X_val.columns, index=X_val.index)

In [7]:
# Here we train one classifier per condition, try 3 model types and keep the best

models = {
    'LogisticRegression': lambda: LogisticRegression(class_weight='balanced', max_iter=1000),
    'RandomForest': lambda: RandomForestClassifier(class_weight='balanced', n_estimators=200, random_state=42),
    'XGBoost': lambda: XGBClassifier(eval_metric='logloss', random_state=42),
}

results = {}
best_models = {}

for condition in conditions:
    print(f"\n--- {condition} ---")
    best_score = -1
    best_name = None
    for name, make_model in models.items():
        model = make_model()
        if name == 'LogisticRegression':
            model.fit(X_train_scaled, y_train[condition])
            val_probs = model.predict_proba(X_val_scaled)[:, 1]
        else:
            model.fit(X_train, y_train[condition])
            val_probs = model.predict_proba(X_val)[:, 1]
        auroc = roc_auc_score(y_val[condition], val_probs)
        print(f"{name}: AUROC = {auroc:.3f}")
        if auroc > best_score:
            best_score = auroc
            best_name = name
            best_models[condition] = model
    results[condition] = best_score
    print(f"Best for {condition}: {best_name} ({best_score:.3f})")

print("\n=== Summary ===")
for cond, score in results.items():
    print(f"{cond}: {score:.3f}")


--- Atelectasis ---
LogisticRegression: AUROC = 0.662
RandomForest: AUROC = 0.660
XGBoost: AUROC = 0.606
Best for Atelectasis: LogisticRegression (0.662)

--- Effusion ---
LogisticRegression: AUROC = 0.639
RandomForest: AUROC = 0.637
XGBoost: AUROC = 0.630
Best for Effusion: LogisticRegression (0.639)

--- Infiltration ---
LogisticRegression: AUROC = 0.604
RandomForest: AUROC = 0.599
XGBoost: AUROC = 0.636
Best for Infiltration: XGBoost (0.636)

--- Nodule ---
LogisticRegression: AUROC = 0.545
RandomForest: AUROC = 0.553
XGBoost: AUROC = 0.526
Best for Nodule: RandomForest (0.553)

--- Pneumothorax ---
LogisticRegression: AUROC = 0.579
RandomForest: AUROC = 0.632
XGBoost: AUROC = 0.580
Best for Pneumothorax: RandomForest (0.632)

=== Summary ===
Atelectasis: 0.662
Effusion: 0.639
Infiltration: 0.636
Nodule: 0.553
Pneumothorax: 0.632


In [8]:
#Check feature importances from your RandomForest models (since it won 3 of 5) to see which features are
#actually pulling weight, and which are dead noise:

def get_feature_importance(model, feature_names):
    if hasattr(model, 'feature_importances_'):
        # Tree-based models (RandomForest, XGBoost)
        return pd.Series(model.feature_importances_, index=feature_names)
    elif hasattr(model, 'coef_'):
        # Linear models (LogisticRegression) — use absolute value of coefficients
        return pd.Series(model.coef_[0], index=feature_names).abs()
    else:
        raise ValueError(f"Error occured for {type(model)}")


In [9]:
# Now check every condition's winning model, not just Atelectasis
for condition in conditions:
    print(f"\n--- {condition} (best model: {type(best_models[condition]).__name__}) ---")
    importance = get_feature_importance(best_models[condition], X_train.columns)
    print(importance.sort_values(ascending=False).head(10))
    
# For LogisticRegression conditions specifically, see direction too
# +ve coefficient means higher feature value pushes toward "positive for this condition," 
# -ve means the opposite.
if hasattr(best_models['Atelectasis'], 'coef_'):
    coefs = pd.Series(best_models['Atelectasis'].coef_[0], index=X_train.columns)
    print(coefs.sort_values(ascending=False))  # most positive = pushes toward "has condition"


--- Atelectasis (best model: LogisticRegression) ---
upper_kurtosis    1.005756
lower_mean        0.778714
lower_std         0.726927
hog_mean          0.623796
middle_skew       0.519652
lower_skew        0.436928
middle_std        0.360664
lower_kurtosis    0.351745
num_blobs         0.313100
upper_mean        0.289801
dtype: float64

--- Effusion (best model: LogisticRegression) ---
hog_mean            0.971008
middle_kurtosis     0.838719
lower_mean          0.820036
glcm_homogeneity    0.720445
lower_std           0.564679
middle_mean         0.406081
glcm_contrast       0.366912
lower_skew          0.318710
upper_skew          0.318346
lower_kurtosis      0.301526
dtype: float64

--- Infiltration (best model: XGBClassifier) ---
upper_std           0.056634
glcm_energy         0.054592
glcm_homogeneity    0.053198
lower_skew          0.053086
middle_skew         0.053049
middle_std          0.051651
hog_std             0.051532
glcm_contrast       0.051386
upper_skew          0.0

### Point 2 — drop the weakest features for the flat-importance conditions, to simplify the model and strengthen the AUROC score without compromising performance.

In [10]:
# For each condition, keep only features above a minimum importance threshold, re-train, compare AUROC
threshold = 0.02
for condition in ['Infiltration', 'Nodule', 'Pneumothorax']:
    importance = get_feature_importance(best_models[condition], X_train.columns)
    keep_features = importance[importance > threshold].index.tolist()
    print(f"{condition}: keeping {len(keep_features)} of {len(X_train.columns)} features")

Infiltration: keeping 20 of 21 features
Nodule: keeping 20 of 21 features
Pneumothorax: keeping 20 of 21 features


#### Point - 2 improved..

#### A fixed threshold=0.02 didn't cut anything because the importances never dropped that low. Instead of guessing at a threshold, we used a relative cutoff — keep only the top N features per condition, which guarantees an actual reduction regardless of the scale of importance values:

In [11]:
def select_top_features(model, feature_names, top_n=10):
    importance = get_feature_importance(model, feature_names)
    return importance.sort_values(ascending=False).head(top_n).index.tolist()

reduced_features = {}
for condition in ['Infiltration', 'Nodule', 'Pneumothorax']:
    top_features = select_top_features(best_models[condition], X_train.columns, top_n=10)
    reduced_features[condition] = top_features
    print(f"{condition}: reduced to {len(top_features)} features -> {top_features}")

Infiltration: reduced to 10 features -> ['upper_std', 'glcm_energy', 'glcm_homogeneity', 'lower_skew', 'middle_skew', 'middle_std', 'hog_std', 'glcm_contrast', 'upper_skew', 'middle_kurtosis']
Nodule: reduced to 10 features -> ['hog_std', 'symmetry_score', 'num_blobs', 'mean_blob_size', 'glcm_contrast', 'upper_kurtosis', 'upper_std', 'middle_kurtosis', 'hog_mean', 'glcm_energy']
Pneumothorax: reduced to 10 features -> ['symmetry_score', 'lower_std', 'lower_mean', 'middle_std', 'glcm_contrast', 'upper_std', 'lower_skew', 'middle_kurtosis', 'mean_blob_size', 'hog_std']


In [12]:
#Then actually retrain on just those features and compare AUROC, to prove the reduction didn't hurt performance:
#This gives you a real before/after: if reduced-feature AUROC is close to the full-feature score, you've got a genuine, demonstrable 
#interpretability win — a much simpler model for basically the same performance.

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

reduction_results = {}
for condition in ['Infiltration', 'Nodule', 'Pneumothorax']:
    cols = reduced_features[condition]
    model = RandomForestClassifier(class_weight='balanced', n_estimators=200, random_state=42)
    model.fit(X_train[cols], y_train[condition])
    val_probs = model.predict_proba(X_val[cols])[:, 1]
    auroc = roc_auc_score(y_val[condition], val_probs)
    reduction_results[condition] = auroc
    print(f"{condition}: reduced-feature AUROC = {auroc:.3f} (full-feature was {results[condition]:.3f})")

Infiltration: reduced-feature AUROC = 0.600 (full-feature was 0.636)
Nodule: reduced-feature AUROC = 0.524 (full-feature was 0.553)
Pneumothorax: reduced-feature AUROC = 0.586 (full-feature was 0.632)


#### Since "baseline" should represent performance before blob features were added, the cleanest and most reproducible fix is to just retrain quickly using only your non-blob columns:

In [13]:
blob_cols = ['num_blobs', 'max_blob_size', 'mean_blob_size']
non_blob_cols = [c for c in X_train.columns if c not in blob_cols]

baseline_results = {}
for condition in conditions:
    model = RandomForestClassifier(class_weight='balanced', n_estimators=200, random_state=42)
    model.fit(X_train[non_blob_cols], y_train[condition])
    val_probs = model.predict_proba(X_val[non_blob_cols])[:, 1]
    baseline_results[condition] = roc_auc_score(y_val[condition], val_probs)

pd.DataFrame(baseline_results.items(), columns=['Condition', 'AUROC']).to_csv('Outputs/baseline_scores.csv', index=False)
print(pd.DataFrame(baseline_results.items(), columns=['Condition', 'AUROC']))

      Condition     AUROC
0   Atelectasis  0.655989
1      Effusion  0.625359
2  Infiltration  0.608204
3        Nodule  0.556147
4  Pneumothorax  0.610590


In [14]:
importance_records = []
for condition in conditions:
    importance = get_feature_importance(best_models[condition], X_train.columns)
    for feat, val in importance.items():
        importance_records.append({'condition': condition, 'feature': feat, 'importance': val})

pd.DataFrame(importance_records).to_csv('Outputs/feature_importance.csv', index=False)

#### Point 3 — We needed to calibrate the final chosen models so output probabilities are meaningful and useful

In [15]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_auc_score

calibrated_models = {}
for condition in conditions:
    model_type = type(best_models[condition]).__name__
    cal_model = CalibratedClassifierCV(best_models[condition], method='isotonic', cv=3)
    
    if model_type == 'LogisticRegression':
        cal_model.fit(X_train_scaled, y_train[condition])
        val_probs = cal_model.predict_proba(X_val_scaled)[:, 1]
    else:
        cal_model.fit(X_train, y_train[condition])
        val_probs = cal_model.predict_proba(X_val)[:, 1]
    
    calibrated_models[condition] = cal_model
    auroc = roc_auc_score(y_val[condition], val_probs)
    print(f"{condition}: Calibrated AUROC = {auroc:.3f}")

Atelectasis: Calibrated AUROC = 0.653
Effusion: Calibrated AUROC = 0.638
Infiltration: Calibrated AUROC = 0.589
Nodule: Calibrated AUROC = 0.565
Pneumothorax: Calibrated AUROC = 0.599


#### Confirm the blob features actually helped Nodule.

In [16]:
comparison = pd.DataFrame({
    'baseline': pd.read_csv('outputs/baseline_scores.csv').set_index('Condition')['AUROC'],
    'with_blob_features': pd.Series(results)  # from your latest run's `results` dict
})
comparison['improvement'] = comparison['with_blob_features'] - comparison['baseline']
print(comparison)

              baseline  with_blob_features  improvement
Atelectasis   0.655989            0.661549     0.005560
Effusion      0.625359            0.638629     0.013270
Infiltration  0.608204            0.635915     0.027711
Nodule        0.556147            0.553468    -0.002679
Pneumothorax  0.610590            0.632287     0.021697


In [17]:
# Save calibrated scores
pd.DataFrame(
    [(c, roc_auc_score(y_val[c], calibrated_models[c].predict_proba(X_val_scaled if type(best_models[c]).__name__=='LogisticRegression' else X_val)[:,1])) for c in conditions],
    columns=['Condition', 'Calibrated_AUROC']
).to_csv('Outputs/calibrated_scores.csv', index=False)

# Save updated feature comparison
comparison.to_csv('Outputs/baseline_vs_blob_comparison.csv', index=False)

In [18]:
#Final checks to make sure we have the expected outputs without anomalies

check = pd.read_csv('Outputs/train_features.csv')
print(check.shape)
print('num_blobs' in check.columns, 'max_blob_size' in check.columns, 'mean_blob_size' in check.columns)
print(pd.read_csv('Outputs/baseline_vs_blob_comparison.csv'))
print(pd.read_csv('Outputs/calibrated_scores.csv'))

(4484, 22)
True True True
   baseline  with_blob_features  improvement
0  0.655989            0.661549     0.005560
1  0.625359            0.638629     0.013270
2  0.608204            0.635915     0.027711
3  0.556147            0.553468    -0.002679
4  0.610590            0.632287     0.021697
      Condition  Calibrated_AUROC
0   Atelectasis          0.653453
1      Effusion          0.637648
2  Infiltration          0.589114
3        Nodule          0.564622
4  Pneumothorax          0.598827


##  Final Test Set Inference & Predictions Export

In [19]:
# Load test features (already extracted with blob columns in 03_feature_extraction.ipynb)
test_features = pd.read_csv('Outputs/test_features.csv')

# Keep the same feature columns used in training
X_test = test_features[feature_cols]

# Also need a scaled version for whichever conditions used LogisticRegression
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

print("NaNs in X_test:", X_test.isna().sum().sum())
print(X_test.isna().sum()[X_test.isna().sum() > 0])  # which columns specifically

NaNs in X_test: 34
upper_skew          1
upper_kurtosis      1
middle_skew         1
middle_kurtosis     1
lower_skew         15
lower_kurtosis     15
dtype: int64


### Here we found out that X_test also has NaN values (i.e 34 NaN values exactly). 

### The fix here is imputation, not dropping — since dropping should only be done on train data.

In [20]:
# Use the median from your TRAINING data, not the test data itself
# (this avoids "leaking" information from the test set into how you fill gaps)
train_medians = X_train.median()

X_test_filled = X_test.fillna(train_medians)
X_test_scaled_filled = pd.DataFrame(
    scaler.transform(X_test_filled),
    columns=X_test_filled.columns,
    index=X_test_filled.index
)

print("Remaining NaNs in X_test:", X_test_filled.isna().sum().sum())  # should be 0

Remaining NaNs in X_test: 0


In [21]:
predictions = pd.DataFrame()
predictions['Image Index'] = test_features['filename']

for condition in conditions:
    model_type = type(best_models[condition]).__name__
    if model_type == 'LogisticRegression':
        probs = calibrated_models[condition].predict_proba(X_test_scaled_filled)[:, 1]
    else:
        probs = calibrated_models[condition].predict_proba(X_test_filled)[:, 1]
    predictions[condition] = probs

predictions.to_csv('../predictions/predictions.csv', index=False)
print(predictions.shape)  # confirms (1122, 6)
print(predictions.isna().sum().sum())  # confirm 0 NaNs in final output

(1122, 6)
0


####    At the end, we get the balanced accuracy as 50% (exactly) for all disease, as initially, we took threshold = 0.5. That exact 50.0% repeated identically across every single condition is a strong signal — a sign that the model is predicting the same class for every single validation image in each condition.

####    This is actually valuable: AUROC scores (0.55-0.66) show the model is learning some ranking signal, but a flat 50% balanced accuracy means the threshold of 0.5 is wrong for these probabilities

####    Hence, lets pick a threshold based on the actual probability distribution, not a fixed 0.5

In [22]:
from sklearn.metrics import balanced_accuracy_score
from sklearn.metrics import accuracy_score
import numpy as np

best_thresholds = {}

for condition in conditions:
    model_type = type(best_models[condition]).__name__
    val_probs = calibrated_models[condition].predict_proba(X_val_scaled if model_type=='LogisticRegression' else X_val)[:, 1]
    
    best_score = -1
    best_thresh = 0.5
    for thresh in np.arange(0.05, 0.95, 0.05):
        preds = (val_probs >= thresh).astype(int)
        score = balanced_accuracy_score(y_val[condition], preds)
        if score > best_score:
            best_score = score
            best_thresh = thresh
    
    best_thresholds[condition] = best_thresh
    print(f"{condition}: best threshold = {best_thresh:.2f}, balanced accuracy = {best_score*100:.1f}%")

Atelectasis: best threshold = 0.10, balanced accuracy = 63.1%
Effusion: best threshold = 0.10, balanced accuracy = 60.9%
Infiltration: best threshold = 0.15, balanced accuracy = 58.1%
Nodule: best threshold = 0.05, balanced accuracy = 53.6%
Pneumothorax: best threshold = 0.05, balanced accuracy = 57.2%


In [23]:
final_metrics = {}
for condition in conditions:
    model_type = type(best_models[condition]).__name__
    val_probs = calibrated_models[condition].predict_proba(X_val_scaled if model_type=='LogisticRegression' else X_val)[:, 1]
    preds = (val_probs >= best_thresholds[condition]).astype(int)
    
    final_metrics[condition] = {
        'threshold_used': best_thresholds[condition],
        'accuracy': accuracy_score(y_val[condition], preds),
        'balanced_accuracy': balanced_accuracy_score(y_val[condition], preds),
        'auroc': roc_auc_score(y_val[condition], val_probs)
    }

metrics_df = pd.DataFrame(final_metrics).T
print(metrics_df)
metrics_df.to_csv('Outputs/validation_accuracy_vs_auroc.csv')

              threshold_used  accuracy  balanced_accuracy     auroc
Atelectasis             0.10  0.560316           0.630705  0.653453
Effusion                0.10  0.519729           0.609242  0.637648
Infiltration            0.15  0.338219           0.580976  0.589114
Nodule                  0.05  0.196167           0.536153  0.564622
Pneumothorax            0.05  0.703495           0.571972  0.598827


###    Accuracy Percentages

In [24]:
metrics_pct = metrics_df.copy()
for col in ['accuracy', 'balanced_accuracy', 'auroc']:
    metrics_pct[col] = (metrics_df[col] * 100).round(1).astype(str) + '%'

print(metrics_pct)
metrics_pct.to_csv('Outputs/validation_metrics_percentage.csv')

              threshold_used accuracy balanced_accuracy  auroc
Atelectasis             0.10    56.0%             63.1%  65.3%
Effusion                0.10    52.0%             60.9%  63.8%
Infiltration            0.15    33.8%             58.1%  58.9%
Nodule                  0.05    19.6%             53.6%  56.5%
Pneumothorax            0.05    70.3%             57.2%  59.9%
